In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, log_loss

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
drop_cols = ['id', 'alpha', 'delta']
train_df = train_df.drop(columns=drop_cols, errors='ignore')
test_ids = test_df['id']
test_df = test_df.drop(columns=drop_cols, errors='ignore')

In [ ]:
target_le = LabelEncoder()
y = target_le.fit_transform(train_df['class'])
X = train_df.drop(columns=['class'])

In [ ]:
cat_cols = ['spectral_type', 'galaxy_population']
ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = ordinal_enc.fit_transform(X[cat_cols])

In [ ]:
X_test = test_df.copy()
X_test[cat_cols] = ordinal_enc.transform(X_test[cat_cols])

In [ ]:
num_classes = len(target_le.classes_)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), num_classes))
test_preds = np.zeros((len(X_test), num_classes))

In [ ]:
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': num_classes,
    'n_estimators': 1000,
    'learning_rate': 0.05,        # Slightly faster convergence
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',        # Fast histogram method
    'early_stopping_rounds': 30,  # Stops early when score plateaus (Fixes training lag)
    'random_state': 42,
    'n_jobs': -1                  # Uses all CPU cores
}
print("Starting Cross-Validation Training...\n" + "-"*40)

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    print(f"Fold {fold + 1} finished at best iteration: {model.best_iteration}")
    
    # Store validation predictions and accumulate test predictions
    oof_preds[val_idx] = model.predict_proba(X_val)
    test_preds += model.predict_proba(X_test) / kf.n_splits

In [ ]:
oof_classes = np.argmax(oof_preds, axis=1)
print("-" * 40)
print(f"Overall OOF Accuracy: {accuracy_score(y, oof_classes):.4f}")
print(f"Overall OOF Multi-LogLoss: {log_loss(y, oof_preds):.4f}")

In [ ]:
final_test_classes = np.argmax(test_preds, axis=1)
predicted_class_names = target_le.inverse_transform(final_test_classes)

submission = pd.DataFrame({
    'id': test_ids,
    'class': predicted_class_names
})

submission.to_csv('predictions3.csv', index=False)
print("\nPredictions successfully saved to 'predictions3.csv'!")
submission.head()